# Stretch: more questions, worked solutions

The answers to the stretch notebook. Try each one before looking.

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

## Stretch 1: how many artists are still active?

Three ways to ask, and they give 39, 34 and 34:

In [ ]:
run("SELECT COUNT(still_active) AS active FROM artists")

In [ ]:
run("SELECT SUM(still_active) AS active FROM artists")

In [ ]:
run("SELECT COUNT(*) AS active FROM artists WHERE still_active = 1")

**The first one is wrong.** `COUNT(still_active)` does not count the
artists who are active. It counts the artists whose `still_active` is
**not NULL**, and that includes every `0`. "Is there a value?" and "is
the value 1?" are different questions.

The other two are both right. `SUM` works because the column is already a
0 or a 1, so adding it up counts the ones. Homework question 9 uses the
same trick on `skipped`.

The second part:

In [ ]:
run("SELECT COUNT(*) AS not_active FROM artists WHERE still_active != 1")

34 active and 5 not active makes 39. There are 40 artists, so one is
missing from both answers.

In [ ]:
run("SELECT artist_name, still_active FROM artists WHERE still_active IS NULL")

Iron Fernway's `still_active` is unknown. Is unknown equal to 1? Unknown.
Is it different from 1? Also unknown. So `WHERE` drops the row both
times, and the artist disappears from "active" **and** from "not active".

That is how NULL catches people in real reports: two numbers that should
add up to the total, and do not. A count that does not add up is
always worth chasing.

## Stretch 2: tracks played on every device

35 of the 168 tracks have been played on all five devices.

In [ ]:
run("""
SELECT track_name,
       COUNT(DISTINCT device) AS devices,
       COUNT(*)               AS plays
FROM plays
GROUP BY track_name
HAVING COUNT(DISTINCT device) = 5
ORDER BY plays DESC
""")

Now only the plays that were not skipped. `skipped = 0` is a question
about **each play**, so it goes in the `WHERE`, before the grouping:

In [ ]:
run("""
SELECT track_name,
       COUNT(DISTINCT device) AS devices,
       COUNT(*)               AS plays
FROM plays
WHERE skipped = 0
GROUP BY track_name
HAVING COUNT(DISTINCT device) = 5
ORDER BY plays DESC
""")

30 tracks. Five drop out, because on at least one device they were only
ever skipped: Velvet Season, for example, was never listened to in the
car.

**The trap version.** Run this, with the condition moved to `HAVING`:

In [ ]:
run("""
SELECT track_name, COUNT(DISTINCT device) AS devices
FROM plays
GROUP BY track_name
HAVING COUNT(DISTINCT device) = 5 AND skipped = 0
""")

No error, 29 rows, and it looks like an answer. It is not. Blue Ferry
and Golden Garden are on the list, and both dropped out of the right
answer.

By `HAVING` time each track is **one group**, made of many plays. Some
were skipped and some were not, so "the group's `skipped`" means
nothing. SQLite picks one play's value and says nothing about it. That
is the `GROUP BY` trap from the slides, hiding in a `HAVING`.

The test: a condition about one row goes in `WHERE`. A condition about a
whole group (a count, an average, a sum) goes in `HAVING`.

## Stretch 3: do I listen more at the weekend?

In [ ]:
run("""
SELECT CASE
           WHEN strftime('%w', played_at) IN ('0', '6') THEN 'weekend'
           ELSE 'weekday'
       END                        AS part_of_week,
       COUNT(*)                   AS plays,
       COUNT(DISTINCT played_at)  AS days,
       ROUND(1.0 * COUNT(*) / COUNT(DISTINCT played_at), 2) AS plays_per_day
FROM plays
GROUP BY part_of_week
""")

Weekdays have 1,573 plays and weekends 610. That looks like a clear
answer: far more listening on weekdays.

**It is not a fair comparison.** A week has five weekdays and two weekend
days, so weekdays win on totals whatever the habits are. Per day it is
3.19 against 3.23: no difference at all.

Whenever two groups are different sizes, compare a rate, not a total.
Totals answer "where did most of it happen", rates answer "where is it
more likely". The question here was the second one.

Two details:

* `COUNT(DISTINCT played_at)` counts days **with at least one play**.
  Days with no listening do not appear in `plays`, so they cannot be
  counted. Fine here, worth knowing.
* `1.0 *` is the integer-division fix: without it, `1573 / 493` is `3`.

## Stretch 4: who did I not play in 2025?

The obvious version first. It runs, it looks fine, and it is wrong.

In [ ]:
run("""
SELECT a.artist_name, COUNT(p.play_id) AS plays_2025
FROM artists a
LEFT JOIN plays p ON p.artist_id = a.artist_id
WHERE p.played_at >= '2025-01-01'
GROUP BY a.artist_id
ORDER BY plays_2025, a.artist_name
""", limit=5)

**35 rows, and nobody has 0 plays.** We asked for 40 artists.

Why: `WHERE` runs *after* the join. For an artist with no 2025 plays, the
left join produces rows whose `played_at` is either a 2024 date or `NULL`.
The `WHERE` throws all of those away, so the artist vanishes. The left join
did its job and the `WHERE` undid it. A filter on the right-hand table in
the `WHERE` quietly turns a left join back into an inner join.

The fix: move the condition into the `ON`. Now it is part of the matching
rule ("match this artist's plays, *and only the 2025 ones*"), and an artist
with no match is still kept, with `NULL`s.

In [ ]:
run("""
SELECT a.artist_name, COUNT(p.play_id) AS plays_2025
FROM artists a
LEFT JOIN plays p
       ON p.artist_id = a.artist_id
      AND p.played_at >= '2025-01-01'
GROUP BY a.artist_id
ORDER BY plays_2025, a.artist_name
""", limit=7)

All 40 artists are back, and five of them have 0 plays in 2025.

Now only the ones with no match:

In [ ]:
run("""
SELECT a.artist_name, a.genre
FROM artists a
LEFT JOIN plays p
       ON p.artist_id = a.artist_id
      AND p.played_at >= '2025-01-01'
WHERE p.play_id IS NULL
ORDER BY a.artist_name
""")

The same five. This shape, a left join followed by `WHERE right.key IS
NULL`, is the standard way to find "things with no match": customers with
no orders, users who never logged in, and so on.

**Why is this `WHERE` fine, when the first one was not?** Both run after
the join. The first one kept rows *with* a 2025 date, which threw away
exactly the unmatched rows. This one keeps *only* the unmatched rows,
which is what we wanted. A `WHERE` on the right-hand table is not always
wrong; you just have to know which rows it is keeping.

Two of the five, Iron Fernway and Gravel Choir, have never been played at
all. The other three were played in 2024 and then dropped. Stretch 6
separates them.

## Stretch 5: spot the bug

The broken query, run:

In [ ]:
run("""
SELECT a.genre,
       COUNT(w.award_id) AS awards,
       COUNT(p.play_id)  AS plays
FROM artists a
LEFT JOIN awards w USING (artist_id)
LEFT JOIN plays  p USING (artist_id)
GROUP BY a.genre
ORDER BY a.genre
""")

Folk claims 1,095 awards. There are only 33 awards in the whole table.

Both columns fan out, and they multiply each other. Every award meets
every play of the same artist, so Fjord & Flint alone contributes
3 × 237 = 711 rows. In that block, `COUNT(w.award_id)` counts 711 awards
and `COUNT(p.play_id)` counts 711 plays.

The nasty part is that the damage is uneven. Afrobeat and Pop have the
right play counts (180 and 302), because their artists have at most one
award each. Jazz is out by exactly 1. If you had spot-checked one of
those genres, you would have concluded the query was fine.

In [ ]:
run("""
SELECT a.genre,
       COUNT(DISTINCT w.award_id) AS awards,
       COUNT(DISTINCT p.play_id)  AS plays
FROM artists a
LEFT JOIN awards w USING (artist_id)
LEFT JOIN plays  p USING (artist_id)
GROUP BY a.genre
ORDER BY a.genre
""")

Electronic 10 and Folk 9 awards, which add up with the rest to 33. The
plays add up to 2,183, and they match the per-genre counts from the live
exercise.

**Why does `DISTINCT` work here, when `SUM(DISTINCT)` did not?** Because
`award_id` and `play_id` are keys: every real award has a different id.
The only way to see the same id twice is to see the same row twice, so
removing repeats removes exactly the copies the join made and nothing
else.

`minutes_played` is a measurement, not a key. Two different plays can last
4.65 minutes each, so `SUM(DISTINCT minutes_played)` throws away real
plays as well as copies.

**Rule:** `DISTINCT` on a key is a safe repair for a count. `DISTINCT` on
a value never is.

**Harder:** to add minutes as well, `DISTINCT` cannot help, so it is back
to aggregating first. Same recipe as the live exercise, at genre level:

In [ ]:
run("""
WITH plays_per_artist AS (
    SELECT artist_id,
           COUNT(*)            AS plays,
           SUM(minutes_played) AS minutes
    FROM plays
    GROUP BY artist_id
),
awards_per_artist AS (
    SELECT artist_id, COUNT(*) AS awards
    FROM awards
    GROUP BY artist_id
)
SELECT a.genre,
       COALESCE(SUM(aw.awards), 0)           AS awards,
       COALESCE(SUM(pp.plays), 0)            AS plays,
       ROUND(COALESCE(SUM(pp.minutes), 0), 1) AS minutes
FROM artists a
LEFT JOIN plays_per_artist  pp USING (artist_id)
LEFT JOIN awards_per_artist aw USING (artist_id)
GROUP BY a.genre
ORDER BY a.genre
""")

Same awards, same plays as the `DISTINCT` version, so two different
methods agree. That agreement is the check.

## Stretch 6: 2024 against 2025

Three CTEs, each building on the ones above it.

In [ ]:
run("""
WITH minutes_2024 AS (
    SELECT artist_id, ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    WHERE played_at < '2025-01-01'
    GROUP BY artist_id
),
minutes_2025 AS (
    SELECT artist_id, ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    WHERE played_at >= '2025-01-01'
    GROUP BY artist_id
),
compared AS (
    SELECT a.artist_name,
           COALESCE(y1.minutes, 0.0) AS min_2024,
           COALESCE(y2.minutes, 0.0) AS min_2025
    FROM artists a
    LEFT JOIN minutes_2024 y1 USING (artist_id)
    LEFT JOIN minutes_2025 y2 USING (artist_id)
)
SELECT artist_name, min_2024, min_2025,
       ROUND(min_2025 - min_2024, 1) AS change
FROM compared
ORDER BY change DESC
""", limit=40)

Mireille Basset is the biggest riser (+71.7 minutes) and Glass Tram the
biggest faller (-84.0). That drop cost Glass Tram the top spot: it was my
most-listened artist in 2024, and DJ Kompas overtook it in 2025. The total fell from 4596.6 minutes in 2024 to 4383.8 in 2025.

Why each piece is there:

* **Aggregate first.** Each year is reduced to one row per artist before
  anything is joined, so nothing can fan out.
* **Start from `artists` and left join both years.** If you join
  `minutes_2024` to `minutes_2025` with an inner join you get 35 rows, and
  the three artists who dropped to zero vanish. They are exactly the ones
  the question is about.
* **`COALESCE(..., 0.0)`.** Neon Harbour has no 2025 row, so its 2025
  minutes are `NULL`, and `NULL - 23.1` is `NULL`, not `-23.1`. Without
  `COALESCE` its change would be blank and it would sink to the bottom of
  the list, looking like missing data instead of a real drop.
* **`ROUND` on the change as well.** Subtracting two decimals can give
  something like `2.4000000000000004`. That is how computers store
  decimals, not a mistake in the query.

**Harder:** the artists I stopped listening to. `compared` is already
there, so the answer is one `WHERE` on it.

In [ ]:
run("""
WITH minutes_2024 AS (
    SELECT artist_id, ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    WHERE played_at < '2025-01-01'
    GROUP BY artist_id
),
minutes_2025 AS (
    SELECT artist_id, ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    WHERE played_at >= '2025-01-01'
    GROUP BY artist_id
),
compared AS (
    SELECT a.artist_name,
           COALESCE(y1.minutes, 0.0) AS min_2024,
           COALESCE(y2.minutes, 0.0) AS min_2025
    FROM artists a
    LEFT JOIN minutes_2024 y1 USING (artist_id)
    LEFT JOIN minutes_2025 y2 USING (artist_id)
)
SELECT artist_name, min_2024
FROM compared
WHERE min_2024 > 0 AND min_2025 = 0
ORDER BY min_2024 DESC
""")

Neon Harbour, Amara Diallo and Odessa Brass: the three from stretch 4 that
were not "never played". This is what CTEs buy you. Once a step has a
name, the next question is a small change at the bottom, not a new query.

## Stretch 7: the same fix, three ways

**A nested subquery.** The `per_artist` query moves into the `FROM`, in
brackets, and gets its name with `AS s` at the end instead of `WITH` at the
top. Identical result.

In [ ]:
run("""
SELECT a.artist_name, s.plays, s.minutes, COUNT(w.award_id) AS awards
FROM (
    SELECT artist_id,
           COUNT(*)                      AS plays,
           ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    GROUP BY artist_id
) AS s
JOIN artists a USING (artist_id)
LEFT JOIN awards w USING (artist_id)
GROUP BY a.artist_id
ORDER BY s.minutes DESC
LIMIT 3
""")

Fjord & Flint 846.2 again. It works, and it reads inside-out: you have to
find the middle, read that first, then read the outside. With one subquery
that is fine. With three nested inside each other it is not, which is why
the session taught the CTE.

**A subquery after `IN`.** 240 plays either way.

In [ ]:
run("""
SELECT COUNT(*) AS dutch_plays
FROM plays
WHERE artist_id IN (SELECT artist_id FROM artists WHERE country = 'Netherlands')
""")

In [ ]:
run("""
SELECT COUNT(*) AS dutch_plays
FROM plays p
JOIN artists a USING (artist_id)
WHERE a.country = 'Netherlands'
""")

The `IN` version cannot fan out, because it never joins anything: each play
is either in the list or not. That makes it a safe way to ask "plays by
artists who have an award", where a join to `awards` would multiply the
rows. The join version is the one to use when you also want columns from
the other table, like the artist's name.

**A view.** `TEMP` keeps it in this connection only, so `music.db` is not
changed.

In [ ]:
con.execute("DROP VIEW IF EXISTS artist_summary")
con.execute("""
CREATE TEMP VIEW artist_summary AS
SELECT a.artist_id,
       a.artist_name,
       a.country,
       a.genre,
       COUNT(p.play_id)                AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes
FROM artists a
LEFT JOIN plays p USING (artist_id)
GROUP BY a.artist_id
""")
print("view created")

In [ ]:
run("SELECT artist_name, genre FROM artist_summary WHERE plays = 0")

In [ ]:
run("""
SELECT country, ROUND(SUM(minutes), 1) AS minutes
FROM artist_summary
GROUP BY country
ORDER BY minutes DESC
LIMIT 3
""")

Iron Fernway and Gravel Choir, and Germany 1285.1, Netherlands 1265.2,
Norway 1184.0: the same as question 1 of the joins exercise. The view was
defined once, correctly, with `LEFT JOIN` and `COUNT(p.play_id)`, and every
query on top of it inherits that.

**When to use which:**

* **CTE**: the default. One query, read top to bottom.
* **Nested subquery**: fine for something short, like the `IN` list.
* **View**: when the same join keeps coming back across many queries, or
  other people need it. Define it once and nobody gets the `COUNT(*)` wrong
  again.

In [ ]:
con.execute("DROP VIEW artist_summary")
print("view dropped")

## Stretch 8: the top artist in every genre

In [ ]:
run("""
WITH per_artist AS (
    SELECT artist_id, SUM(minutes_played) AS minutes
    FROM plays
    GROUP BY artist_id
),
ranked AS (
    SELECT a.genre,
           a.artist_name,
           ROUND(s.minutes, 1) AS minutes,
           RANK() OVER (PARTITION BY a.genre ORDER BY s.minutes DESC) AS position
    FROM per_artist s
    JOIN artists a USING (artist_id)
)
SELECT genre, artist_name, minutes
FROM ranked
WHERE position = 1
ORDER BY minutes DESC
""")

Eight genres, eight winners. Glass Tram (1175.1) and Fjord & Flint
(846.2) are the same numbers as in notebook 03, which is a free check.

**Why can the `WHERE position = 1` not go in the same `SELECT` as the
`RANK()`?** SQLite refuses with `misuse of aliased window function
position`. Think back to the execution order slide: `WHERE` runs before
`SELECT`. When `WHERE` runs, `position` does not exist yet.
Putting the ranking in a CTE first means it is an ordinary column by the
time the outer `WHERE` sees it.

**Why not `GROUP BY genre` with `MAX(minutes)`?** That tells you the
winning number, not who won. And it cannot do the next part at all.

**Harder:** the top two per genre is one character.

In [ ]:
run("""
WITH per_artist AS (
    SELECT artist_id, SUM(minutes_played) AS minutes
    FROM plays
    GROUP BY artist_id
),
ranked AS (
    SELECT a.genre,
           a.artist_name,
           ROUND(s.minutes, 1) AS minutes,
           RANK() OVER (PARTITION BY a.genre ORDER BY s.minutes DESC) AS position
    FROM per_artist s
    JOIN artists a USING (artist_id)
)
SELECT genre, position, artist_name, minutes
FROM ranked
WHERE position <= 2
ORDER BY genre, position
""", limit=16)

Electronic is the close race: Glass Tram 1175.1, DJ Kompas 1169.2, less
than six minutes apart over two years. Pop is the opposite: Sara Lindqvist
768.6, and second place only 85.9.

If two artists tied, `RANK()` would give them both 1 and skip 2.
`ROW_NUMBER()` would number them 1 and 2 anyway, picking one at random.
Which one you want depends on the question, again.

In [ ]:
con.close()
print("done")